In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
import numpy as np
import pandas as pd
from sklearn import compose, impute, metrics, model_selection
from sklearn import preprocessing, pipeline, tree

## Define a custom scoring function

In [ ]:
root_mean_square_log_error_scoring = metrics.make_scorer(
    metrics.root_mean_squared_log_error,
    greater_is_better=False
)

## Load the data

In [ ]:
train_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/train.csv",
        index_col="Id"
    )
)

In [ ]:
train_df.info()

## Prepare the data

In [ ]:
train_features_df = train_df.drop("SalePrice", axis=1)
train_target = train_df.loc[:, "SalePrice"]

In [ ]:
features_preparation_pipeline = pipeline.make_pipeline(
    compose.make_column_transformer(
        (
            preprocessing.OneHotEncoder(
                categories="auto",
                dtype=np.uint8,
                handle_unknown="infrequent_if_exist",
                max_categories=None,
                min_frequency=None,
                sparse_output=False,
            ),
            [
                "MSSubClass",
                "MSZoning",
                "Street",
                "Alley",
                "LandContour",
                "LotConfig",
                "Neighborhood",
                "Condition1",
                "Condition2",
                "BldgType",
                "HouseStyle",
                "RoofStyle",
                "RoofMatl",
                "Exterior1st",
                "Exterior2nd",
                "MasVnrType",
                "Foundation",
                "Heating",
                "GarageType",
                "Fence",
                "MiscFeature",
                "SaleType",
                "SaleCondition",
            ]
        ),
        (
            impute.SimpleImputer(
                strategy="constant",
                fill_value=0.0,
            ),
            [
                "LotFrontage",
                "MasVnrArea",
                "BsmtFinSF1",
                "BsmtFinSF2",
                "BsmtUnfSF",
                "TotalBsmtSF",
                "BsmtFullBath",
                "BsmtHalfBath",
                "Fireplaces",
                "GarageCars",
                "GarageArea",
                "WoodDeckSF",
                "OpenPorchSF",
                "EnclosedPorch",
                "3SsnPorch",
                "ScreenPorch",
                "PoolArea",
                "MiscVal",
            ]
        ),
        (
            impute.SimpleImputer(
                strategy="median",
            ),
            [
                "LotArea",
                "1stFlrSF",
                "2ndFlrSF",
                "LowQualFinSF",
                "GrLivArea",
            ]
        ),
        (
            preprocessing.OrdinalEncoder(
                categories=[
                    ["IR3", "IR2", "IR1", "Reg"],
                    ["ELO", "NoSeWa", "NoSewr", "AllPub"],
                    ["Sev", "Mod", "Gtl"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["No", "Mn", "Av", "Gd"],
                    ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
                    ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["N", "Y"],
                    ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Sal", "Sev", "Maj2", "Maj1", "Min2", "Min1", "Typ"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    list(range(1900, 2011, 1)),
                    ["Unf", "RFn", "Fin"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["N", "P", "Y"],
                    ["Fa", "TA", "Gd", "Ex"],
                ],
                dtype=np.int8,
                encoded_missing_value=-1,
                handle_unknown="use_encoded_value",
                max_categories=None,
                min_frequency=None,
                unknown_value=-1,
            ),
            [
                "LotShape",
                "Utilities",
                "LandSlope",
                "ExterQual",
                "ExterCond",
                "BsmtQual",
                "BsmtCond",
                "BsmtExposure",
                "BsmtFinType1",
                "BsmtFinType2",
                "HeatingQC",
                "CentralAir",
                "Electrical",
                "KitchenQual",
                "Functional",
                "FireplaceQu",
                "GarageYrBlt",
                "GarageFinish",
                "GarageQual",
                "GarageCond",
                "PavedDrive",
                "PoolQC",
            ]
        ),
        (
            impute.SimpleImputer(
                strategy="most_frequent"
            ),
            [
                "OverallQual",
                "OverallCond",
                "YearBuilt",
                "YearRemodAdd",
                "FullBath",
                "HalfBath",
                "BedroomAbvGr",
                "KitchenAbvGr",
                "TotRmsAbvGrd",
                "MoSold",
                "YrSold",
            ]
        ),
        remainder="drop",
        n_jobs=-1,
        verbose_feature_names_out=False,
    )
).set_output(transform="pandas")

## Fit an unregularized decision tree

In [ ]:
_prng = np.random.RandomState(42)

ml_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    tree.DecisionTreeRegressor(
        random_state=_prng
    ),
)

_ = ml_pipeline.fit(
    train_features_df,
    train_target
)

In [ ]:
# depth of the default tree
(
    ml_pipeline.named_steps
               .get("decisiontreeregressor")
               .tree_
               .max_depth
)

In [ ]:
# number of leaves in the default tree
(
    ml_pipeline.named_steps
               .get("decisiontreeregressor")
               .tree_
               .n_leaves
)

### Evaluate on the training data

In [ ]:
train_prediction = ml_pipeline.predict(train_features_df)

train_rmsle = metrics.root_mean_squared_log_error(
    train_target,
    train_prediction,
)
print(f"Training RMSLE: {train_rmsle}")

### Evaluate using cross-validation

In [ ]:
_prng = np.random.RandomState(42)

cv_splitter = model_selection.KFold(
    n_splits=4,
    shuffle=True,
    random_state=_prng,
)
cv_splits = cv_splitter.split(train_features_df, train_target)

cv_neg_rmsles = model_selection.cross_val_score(
    ml_pipeline,
    train_features_df,
    train_target,
    cv=cv_splits,
    n_jobs=-1,
    scoring=root_mean_square_log_error_scoring,
)

print(f"Average CV RMSLE: {np.abs(cv_neg_rmsles).mean()}")

## Structural regularization via constraints

In [ ]:
prepared_train_features_df = (
    features_preparation_pipeline.fit_transform(train_features_df)
)

In [ ]:
features = prepared_train_features_df.columns
constraints = np.zeros(len(features))

for i, feature in enumerate(features):
    if feature in ["OverallQual", "OverallCond", "CentralAir", "Electrical", "KitchenQual", "Functional"]:
        constraints[i] = 1
    elif feature in [] :
        constraints[i] = -1
    else:
        pass
    

In [ ]:
_prng = np.random.RandomState(42)

constrained_decision_tree_regressor = tree.DecisionTreeRegressor(
    monotonic_cst=constraints,
    random_state=_prng
)

_ = constrained_decision_tree_regressor.fit(
    prepared_train_features_df,
    train_target
)

### Evaluate on the training data

In [ ]:
train_prediction = (
    constrained_decision_tree_regressor.predict(
        prepared_train_features_df
    )
)

train_rmsle = metrics.root_mean_squared_log_error(
    train_target,
    train_prediction,
)
print(f"Training RMSLE: {train_rmsle}")

### Evaluate using cross-validation

In [ ]:
_prng = np.random.RandomState(42)

cv_splitter = model_selection.KFold(
    n_splits=4,
    shuffle=True,
    random_state=_prng,
)
cv_splits = cv_splitter.split(
    prepared_train_features_df,
    train_target
)

cv_neg_rmsles = model_selection.cross_val_score(
    constrained_decision_tree_regressor,
    prepared_train_features_df,
    train_target,
    cv=cv_splits,
    n_jobs=-1,
    scoring=root_mean_square_log_error_scoring,
)

print(f"Average CV RMSLE: {np.abs(cv_neg_rmsles).mean()}")

## Additional regularization via HPO

In [ ]:
_prng = np.random.RandomState(42)

param_grid = [
    {
        "max_depth": [None, 2, 4, 8, 16],
        "max_leaf_nodes": [None, 10, 100, 1000],
        "min_samples_leaf": [1, 0.01, 0.02, 0.04, 0.08],
        "min_samples_split": [2, 0.02, 0.04, 0.08, 0.16],
        "max_features": [None, 0.25, 0.5, 0.75],
    }
]

cv_splitter = model_selection.KFold(
    n_splits=4,
    shuffle=True,
    random_state=_prng,
)

cv_splits = cv_splitter.split(
    prepared_train_features_df,
    train_target
)

regularized_decision_tree_regressor = model_selection.GridSearchCV(
    constrained_decision_tree_regressor,
    param_grid,
    cv=cv_splits,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    scoring=root_mean_square_log_error_scoring
)

In [ ]:
_ = regularized_decision_tree_regressor.fit(
    prepared_train_features_df,
    train_target
)

In [ ]:
regularized_decision_tree_regressor.best_params_

In [ ]:
np.abs(regularized_decision_tree_regressor.best_score_)

In [ ]:
(
    pd.DataFrame
      .from_dict(
          regularized_decision_tree_regressor.cv_results_
      ).sort_values(
          "rank_test_score"
      )
)

In [ ]:
regularized_decision_tree_regressor.best_estimator_.tree_.n_leaves

## Prune the regularized tree

In [ ]:
ccp_path = (
    regularized_decision_tree_regressor.best_estimator_
                                       .cost_complexity_pruning_path(
                                           prepared_train_features_df,
                                           train_target
                                       )
)

In [ ]:
_prng = np.random.RandomState(42)

param_grid = [
    {
        "ccp_alpha": ccp_path.ccp_alphas
    }
]

cv_splitter = model_selection.KFold(
    n_splits=4,
    shuffle=True,
    random_state=_prng,
)

cv_splits = cv_splitter.split(
    prepared_train_features_df,
    train_target
)

pruned_decision_tree_regressor = model_selection.GridSearchCV(
    regularized_decision_tree_regressor.best_estimator_,
    param_grid,
    cv=cv_splits,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    scoring=root_mean_square_log_error_scoring
)

In [ ]:
_ = pruned_decision_tree_regressor.fit(
    prepared_train_features_df,
    train_target
)

In [ ]:
pruned_decision_tree_regressor.best_params_

In [ ]:
np.abs(pruned_decision_tree_regressor.best_score_)

In [ ]:
pruned_decision_tree_regressor.best_estimator_.tree_.n_leaves

## Submit predictions to the competition!

### Load the test features

In [ ]:
test_features_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/test.csv",
        index_col="Id"
    )
)

### Create the test predictions

In [ ]:
prepared_test_features_df = features_preparation_pipeline.transform(
    test_features_df
)

test_prediction = (
    pruned_decision_tree_regressor.predict(
        prepared_test_features_df
    )
)

### Create the submission file

In [ ]:
(
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    ).assign(
        SalePrice=test_prediction
    ).to_csv(
        "submission.csv",
        index=True
    )
)

In [ ]:
%%bash

cat submission.csv | head -n 5